# Deployment latency, stage by stage

Table 6 reports 193 ms/image on CPU and 72 ms on a T4, with no measurement code in
the repository. A reviewer timed the deployed application at 1.2-2.5 s per image.

This notebook times each stage separately on the same weights the Space serves, so
the two figures can be reconciled instead of defended. The deployed pipeline is
detection, axis-aligned cropping, script-specific PaddleOCR recognition, then a
`GoogleTranslator` call per detected line. That last stage is a network request to
an external service, not model inference.

**Attach one dataset:** *Add Input -> Datasets -> `rishiksaisanthosh/dataset-test`* for the MLT
images.

**Accelerator: GPU T4 x1.** GPU and CPU are both measured; one T4 is enough.

Kaggle gives 4 vCPUs against the Space's 2, so the CPU figures here are optimistic
for the deployment and should be reported as measured on this machine.

In [ ]:
N_IMAGES = 30          # timed images; the spread matters as much as the mean
WARMUP = 5             # excluded - first calls pay lazy init and cuDNN autotune
CONF = 0.4
TARGET_LANG = "en"
MEASURE_TRANSLATION = True   # needs Internet on; it is a network call, not compute

In [ ]:
!pip install -q "ultralytics==8.3.189" deep-translator
!pip install -q --timeout 180 --retries 10 paddlepaddle==3.2.0
!pip install -q paddleocr
!git clone -q --branch ablation-mscbam-probe https://github.com/SaiSanthosh1508/End-to-End-Text-Translation-Pipeline.git /kaggle/working/repo
import sys; sys.path.insert(0, "/kaggle/working/repo")

In [ ]:
!cd /kaggle/working/repo && python ablation/install_modules.py

## 1. The weights the Space actually serves

`best.pt` is stored in Git LFS, so a plain clone yields a 133-byte pointer. Pulling
it from the Space guarantees these timings describe the deployed model.

In [ ]:
from pathlib import Path

!wget -q -O /kaggle/working/best.pt https://huggingface.co/spaces/sai-santhosh/Text_Translation_Pipeline/resolve/main/best.pt
weights = Path("/kaggle/working/best.pt")
print(weights, weights.stat().st_size, "bytes")
assert weights.stat().st_size == 21363137, "not the deployed checkpoint"

import torch
from rects_control.detectors import register_pickle_aliases
register_pickle_aliases()
from ultralytics import YOLO

detector = YOLO(str(weights))
params = sum(p.numel() for p in detector.model.parameters())
print(f"{params/1e6:.2f}M parameters")

## 2. Images

In [ ]:
import itertools

candidates = sorted(itertools.islice(
    (p for p in Path("/kaggle/input").rglob("*.jpg") if "images/val" in str(p)),
    N_IMAGES + WARMUP,
))
if not candidates:
    candidates = sorted(itertools.islice(Path("/kaggle/input").rglob("*.jpg"),
                                         N_IMAGES + WARMUP))
assert len(candidates) >= N_IMAGES + WARMUP, f"only {len(candidates)} images found"
print(f"{len(candidates)} images")

## 3. The pipeline stages

Recognition mirrors the deployed application: one PaddleOCR mobile model per script
class, selected by the detector's predicted label, loaded once and reused.

In [ ]:
import cv2
import numpy as np
from paddleocr import TextRecognition
from deep_translator import GoogleTranslator

CLASS_MODELS = {
    0: "arabic_PP-OCRv3_mobile_rec", 1: "en_PP-OCRv3_mobile_rec",
    2: "ch_PP-OCRv4_mobile_rec",     3: "korean_PP-OCRv3_mobile_rec",
    4: "japan_PP-OCRv3_mobile_rec",  5: "bangla_PP-OCRv3_mobile_rec",
    6: "devanagari_PP-OCRv3_mobile_rec", 7: "en_PP-OCRv3_mobile_rec",
}
engines = {}

def recogniser_for(cls):
    name = CLASS_MODELS[int(cls)]
    if name not in engines:
        engines[name] = TextRecognition(model_name=name, device="cpu")
    return engines[name]

def crop(image, quad):
    pts = np.asarray(quad, dtype=int)
    h, w = image.shape[:2]
    y0, y1 = max(0, pts[:, 1].min() - 2), min(h, pts[:, 1].max() + 2)
    x0, x1 = max(0, pts[:, 0].min() - 2), min(w, pts[:, 0].max() + 2)
    return image[y0:y1, x0:x1]

def translate(texts):
    if not (MEASURE_TRANSLATION and texts):
        return texts
    return [GoogleTranslator(source="auto", target=TARGET_LANG).translate(t) for t in texts]

## 4. Measure

Warm-up runs are discarded: the first inference pays lazy model initialisation and
cuDNN autotuning, which is not what a deployed request costs.

In [ ]:
from latency.measure import as_table, summarise, time_image

def run(device):
    detector.to(device)
    classes_seen = []

    def detect(image):
        result = detector.predict(image, conf=CONF, device=device, verbose=False)[0]
        if result.obb is None or len(result.obb) == 0:
            classes_seen.clear()
            return []
        classes_seen[:] = result.obb.cls.cpu().numpy().tolist()
        return list(result.obb.xyxyxyxy.cpu().numpy())

    def recognise(crops):
        out = []
        for c, cls in zip(crops, classes_seen):
            prediction = recogniser_for(cls).predict([c], batch_size=1)
            out.append(prediction[0]["rec_text"].strip())
        return out

    runs = []
    for n, path in enumerate(candidates):
        image = cv2.imread(str(path))
        times = time_image(image, detect, crop, recognise, translate)
        if n >= WARMUP:
            runs.append(times)
    return summarise(runs)

results = {}
for device in ("cuda:0", "cpu"):
    print(f"\n===== {device} =====", flush=True)
    results[device] = run(device)
    print(as_table(results[device]))

## 5. The table for the paper

In [ ]:
print(f"{'stage':12s} {'GPU (T4)':>18s} {'CPU (4 vCPU)':>18s}")
print("-" * 50)
for stage in ("detect", "crop", "recognise", "translate", "total"):
    g, gs = results["cuda:0"][stage]
    c, cs = results["cpu"][stage]
    print(f"{stage:12s} {g:10.1f} +/- {gs:5.1f} {c:10.1f} +/- {cs:5.1f}")
print()
print(f"detection only, GPU: {results['cuda:0']['detect'][0]:.0f} ms")
print(f"detection only, CPU: {results['cpu']['detect'][0]:.0f} ms")
print(f"end-to-end,     CPU: {results['cpu']['total'][0]:.0f} ms")

## How to read this

Compare `detect` against the paper's 72 ms (GPU) and 193 ms (CPU). If they match,
Table 6 was reporting detection only and simply needs relabelling, with an
end-to-end row added.

The reviewer's 1.2-2.5 s should be close to `total`, and `translate` is expected to
dominate it: one network request per detected line, which is neither model
inference nor something the architecture affects. Reporting it separately is the
honest way to answer that comment.